<a href="https://colab.research.google.com/github/kyungjunoh1/LLM-workspace/blob/main/7_%EC%B0%B8%EA%B3%A0_%EB%AC%B8%EC%84%9C_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### GPU 실행

In [ ]:
!pip install llama-index-llms-langchain==0.8.0 -qqq
!pip install llama-index==0.14.8 -qqq
!pip install llama-index-vector-stores-faiss==0.6.0 -qqq
!pip install llama-index-llms-ollama==0.9.0 -qqq
!pip install llama-index-embeddings-huggingface==0.6.1 -qqq
!pip install llama-index-embeddings-ollama==0.8.4 -qqq
!pip install llama-index-embeddings-ollama==0.8.4 -qqq
!pip install langchain==1.2.15 -qqq
!pip install langchain-community==0.4.1 -qqq
#!pip install faiss-cpu==1.13.2 -qqq

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Tuple

#import faiss

# ---------------- LangChain ----------------
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ---------------- LlamaIndex ----------------
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    Settings,
    SimpleDirectoryReader,
    load_index_from_storage
)
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.schema import NodeWithScore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
#from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.llms.ollama import Ollama #올라마를 라마인덱스로 변환기능
#올라마
from langchain_community.chat_models import ChatOllama
from llama_index.embeddings.ollama import OllamaEmbedding

# 올라마 설치

In [ ]:
!ollama --version

In [ ]:
!apt-get update -y
!apt-get install -y curl #curl 설치
!apt-get install -y zstd #zstd : 압축 방식( 올라마 프로그램이 압축되어 있는 형식 )

In [ ]:
#올라마 설치( 올라마 사이트로 받은 파일 코랩에 sh로 설치 )
!curl -fsSL https://ollama.com/install.sh | sh
# 경고는 무시해도 된다.

### 코랩에서 올라마 서버 구동 후 시작

In [ ]:
#코랩에서 서버 실행 명령어
import subprocess
import time

process = subprocess.Popen(["ollama", "serve"])

# 서버가 뜰 시간을 조금 줌
time.sleep(5)

In [ ]:
!ollama --version

In [ ]:
!ollama list

In [ ]:
!ollama pull llama3 #사용할 모델
!ollama pull nomic-embed-text #임베딩 모델

In [ ]:
!ollama list

In [ ]:
!ollama show llama3
# 파라미터 80억개
# quantization Q4 : 4bit 양자화

### 모델 저장

In [ ]:
llm = ChatOllama(model="llama3", temperature=0.3)

# 라우터 설정

In [ ]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """
너는 사용자 질문을 분류하는 라우터다.

카테고리:
1. policy
- 회사 정책
- 인사/HR
- 연차/휴가/병가
- 근태/출퇴근
- 복지
- 비용처리
- 보안 규정
- 내부 규정
- 사내 프로세스
- 회사 운영 기준

2. general
- 일반 상식
- 개발 지식
- 기술 설명
- 자유 질문


규칙:
1. 반드시 아래 두 단어 중 하나만 출력한다.
   - policy
   - general
2. 설명 추가 금지
3. 이유 설명 금지
4. 문장으로 답변 금지
5. 기호(-, :, 번호) 추가 금지
6. 줄바꿈 추가 금지
7. 애매하지만 회사 제도/내부 기준을 묻는 느낌이면 policy로 출력.
8. 출력 예시:
policy
"""),
    ("user","{question}")
])

In [ ]:
router_chain = router_prompt | llm | StrOutputParser()

In [ ]:
def clean_route(text:str) -> str:
    text = text.strip().lower()
    if "policy" in text:
        return "policy"
    return "general"

### 이전 대화 목록 데이터 저장 기능

In [ ]:
!ls ./

In [ ]:
!mkdir ./db

In [ ]:
!ls ./

In [ ]:
DB_DIR = "./db"
HISTORY_PATH = Path(f"{DB_DIR}/chat_history.json")
history = []

In [ ]:
#실제 Database에 저장해서 사용
def save_history(history_data : List[Dict[str, str]]) -> None:
    with open(HISTORY_PATH, "w", encoding="utf-8") as f:
        # ensure_ascii=False => "\uc548\ub155" 이런 값 대신  "안녕" 문자로 직접 저장
        # indent=2 => 저장시 [{...},{...}...] 한 줄이 아닌 들여쓰기로 보기 편하게 저장
        json.dump(history_data, f, ensure_ascii=False, indent=2)

def add_history(role: str, content: str) -> None:
    history.append({"role": role, "content": content})

### 오케스트레이터

In [ ]:
i = 1
def run_chat(user_question : str ) -> dict:
    global i
    # 질문 분류
    route = clean_route(router_chain.invoke({"question": user_question}))
    # policy면 문서 검색
    if route == "policy":
        answer=f"{i}.문서 답변"
    else:
        answer=f"{i}.일반 답변"
    i += 1
    #history 관리
    add_history("user", user_question)
    add_history("assistant", answer)
    save_history(history)

    return {
        "question" : user_question, #사용자 질문
        "route": route, # policy, general 구분된 값
        "answer" : answer # route에 맞는 답변
    }

### test

In [ ]:
questions = [
    "연차 사용 기준이 뭐야?",
    "병가는 어떻게 써?",
    "딥러닝이 뭐야?",
]
for q in questions:
    result = run_chat(q)
    print("=" * 60)
    print("질문:", result["question"])
    print("route:", result["route"])
    print("답변:", result["answer"])

In [ ]:
!ollama ps

In [ ]:
!ls -R ./db

In [ ]:
!cat ./db/chat_history.json

# 일반 답변 기능 생성
#### history 기능 생성

In [ ]:
history

In [ ]:
history[-6:]

### 이전 목록 10개 데이터 하나로 합치는 기능

In [ ]:
MAX_HISTORY_TURNS = 30
def recent_history_to_messages( max_turns: int = MAX_HISTORY_TURNS ) -> List[ChatMessage]:
    sliced = history[-(max_turns * 2) : ] #짝수 설정 * 2
    messages : List[ChatMessage] = []
    for item in sliced:
        if item['role'] == 'user':
            messages.append(ChatMessage(role=MessageRole.USER, content=item['content']))
        elif item['role'] == 'assistant':
            messages.append(ChatMessage(role=MessageRole.ASSISTANT, content=item['content']))
    return messages
recent_history_to_messages() #이전 대화 목록 저장 확인

In [ ]:
general_answer_prompt = """
너는 친절한 AI agent이다.
질문에 대해 한국어로 자연스럽고 이해하기 쉽게 답변해라.
"""

In [ ]:
Settings.llm = llm #chat 기능사용하기 위해 라마인덱스로 변환

### 위에서 저장한 이전 대화 목록 + 현재 목록 합치는 기능

In [ ]:
#히스토리 기능 적용
def call_llm(system_prompt: str, user_content: str) -> str:
    #시스템 msg
    messages = [ChatMessage(role=MessageRole.SYSTEM, content=system_prompt)]
    #이전 대화 기록. assistant
    messages.extend(recent_history_to_messages())
    #사용자 질문 현재 user
    messages.append(ChatMessage(role=MessageRole.USER, content=user_content))

    response = Settings.llm.chat(messages)
    return response.message.content.strip() #strip 양 쪽 공백 삭제
call_llm(general_answer_prompt, "오늘의 날씨")

In [ ]:
#일반 추론 결과
def answer_general(user_question: str, fallback_reason: str = "") -> None:
    answer = call_llm(
        general_answer_prompt,
        user_question
    )

    prefix = "[일반 추론 답변]\n"
    #fallback 내용이 나중에 들어오면 넣어주기 위한 공간(정책자료 없음. 일반지식으로 변경)
    #지금은 fallback가 아니라서 "" 표현함
    if fallback_reason:
        prefix += f"{fallback_reason}\n\n"

    return prefix + answer

In [ ]:
def answer_from_general( question : str ) -> str:
    answer = answer_general( question )
    return answer

### 오케스트레이터

In [ ]:
def run_chat(user_question : str ) -> dict:
    # 질문 분류
    route = clean_route(router_chain.invoke({"question": user_question}))
    # policy면 문서 검색
    if route == "policy":
        answer=f"{i}.문서 답변"
    else:
        # 추론 후 내용 전달로 수정
        answer = answer_from_general(user_question) #"일반 답변"
    #history 관리
    add_history("user", user_question)
    add_history("assistant", answer)
    save_history(history)

    return {
        "question" : user_question, #사용자 질문
        "route": route, # policy, general 구분된 값
        "answer" : answer # route에 맞는 답변
    }

### test

In [ ]:
questions = [
    "연차 사용 기준이 뭐야?",
    "병가는 어떻게 써?",
    "딥러닝이 뭐야?",
]
for q in questions:
    result = run_chat(q)
    print("=" * 60)
    print("질문:", result["question"])
    print("route:", result["route"])
    print("답변:", result["answer"])

In [ ]:
!ls ./db
!cat ./db/chat_history.json

# 백터 디비 설정
### 임베딩 설정
1. json파일 document 변환
2. 임베딩 모델 생성
3. 백터 디비 설정

In [ ]:
#코랩에 문서 불러와서 실행
from google.colab import drive
drive.mount('/content/drive') #코랩 dirve폴더와 구글 드라이브 연동
#코랩 기본 위치 /content 시작
#코랩의 /content/drive 폴더(drive) 생성 -> 구글 드라이브와 연동

In [ ]:
!ls ./drive/MyDrive/colab-workspace/ex01/policy/

In [ ]:
import json
from llama_index.core import Document
DOCS_DIR = "/content/drive/MyDrive/colab-workspace/ex01/policy"

In [ ]:
with open(DOCS_DIR+"/policie.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
data #json형식의 데이터 확인

In [ ]:
data[0].get("id","") #id 키가 없으면 "" 빈 문자열로 처리

#### Document 생성

In [ ]:
documents = []
for item in data:
  # json파일 documents로 바로 변환 가능. 하지만 구조가 없어 검색 품질 떨어짐( metadata, text )
  #documents = SimpleDirectoryReader(DOCS_DIR).load_data()
  doc_id = item.get("id", "")
  category = item.get("category", "")
  title = item.get("title", "")
  content = item.get("content", "")

  text = f"제목: {title}\n카테고리: {category}\n내용: {content}"

  doc = Document(
      text=text,
      metadata={
          "id": doc_id,
          "category": category,
          "title": title
      }
  )
  documents.append(doc)
#documents[0]

### 백터 디비 생성

In [ ]:
!ollama list

In [ ]:
# 임베딩 전역 설정
#Settings.embed_model = OllamaEmbedding( model_name="nomic-embed-text" )

#허깅페이스 임베딩 모델이 지금 데이터에는 잘 검색된다
Settings.embed_model = HuggingFaceEmbedding( model_name="BAAI/bge-m3" )

In [ ]:
index = VectorStoreIndex.from_documents( documents )#전역설정으로 생략. embed_model=embed_model)

In [ ]:
ret01 = index.as_retriever(similarity_top_k=3)
nodes01 = ret01.retrieve("연차 대해 알려줘")
for n in nodes01:
  print(n)

### 구글 드라이브에 백터DB 내용 저장
- document의 내용까지 저장 되어 있어 로드 후 바로 적용 가능

In [ ]:
index.storage_context.persist(persist_dir=f"{DOCS_DIR}/storage")

### 로드 후 검색 확인

In [ ]:
# 다시 불러오기
storage_context = StorageContext.from_defaults(persist_dir=f"{DOCS_DIR}/storage")
index02 = load_index_from_storage(storage_context)

In [ ]:
# 저장 전 내용과 로드한 index 내용이 일치한다. 즉 저장시 document값도 같이 저장 된다
ret02 = index02.as_retriever(similarity_top_k=3)
nodes02 = ret02.retrieve("연차 대해 알려줘")
for n in nodes02:
  print(n)

In [ ]:
os.path.exists(f"{DOCS_DIR}/storage")

In [ ]:
def get_or_create_index():
    if os.path.exists(f"{DOCS_DIR}/storage"):
        return load_vector_store_index()
    return build_vector_store_index()

In [ ]:
def load_vector_store_index():
    storage_context = StorageContext.from_defaults(persist_dir=f"{DOCS_DIR}/storage")
    index = load_index_from_storage(storage_context)
    return index
#load_vector_store_index()

In [ ]:
def build_vector_store_index():
  documents = []
  for item in data:
    # json파일 documents로 바로 변환 가능. 하지만 구조가 없어 검색 품질 떨어짐( metadata, text )
    #documents = SimpleDirectoryReader(DOCS_DIR).load_data()
    doc_id = item.get("id", "")
    category = item.get("category", "")
    title = item.get("title", "")
    content = item.get("content", "")

    text = f"제목: {title}\n카테고리: {category}\n내용: {content}"

    doc = Document(
        text=text,
        metadata={
            "id": doc_id,
            "category": category,
            "title": title
        }
    )
    documents.append(doc)
  index = VectorStoreIndex.from_documents( documents )
  return index
#build_vector_store_index()

### index 생성 또는 로드

In [ ]:
SIMILARITY_TOP_K = 3
index = get_or_create_index()
retriever = index.as_retriever(similarity_top_k=SIMILARITY_TOP_K)

In [ ]:
user_question = "보안 교육 알려줘?"
nodes = retriever.retrieve(user_question)

In [ ]:
for node in nodes:
    print(node)

### 문서 내용이 있는 내용인지 확인
1. 0.5 이상인 문서들과 최고 유사도 점수 확인
2. 문서들 하나로 합처서 추론

In [ ]:
def retrieve_policy_docs(user_question: str) -> tuple:
    # 문서 검색
    nodes = retriever.retrieve(user_question)

    # 유사도 기준 필터링
    postprocessor = SimilarityPostprocessor(similarity_cutoff=0.5)
    filtered_nodes = postprocessor.postprocess_nodes(nodes)

    # 4. 최고 점수 계산
    top_score = 0.0
    if filtered_nodes:
        top_score = filtered_nodes[0].score

    # 5. 반환
    return filtered_nodes, top_score

In [ ]:
nodes , score = retrieve_policy_docs("보안 교육 알려줘?")
for node in nodes:
    print(node)

#### 2.문서내용 하나로 합처서 추론

In [ ]:
def format_nodes(nodes: List[NodeWithScore]) -> str:
    if not nodes:
        return "검색된 문서 없음"

    parts = []
    for i, node in enumerate(nodes, start=1):
        content = node.node.get_content()
        score = node.score if node.score is not None else 0.0
        #추론시 score가 있으면 LLM이 어떤 내용이 중요한지를 판단할 수 있다
        parts.append(f"[문서{i} | score={score:.4f}]\n{content}")
    return "\n\n".join(parts)

In [ ]:
format_nodes(nodes)

### 문서 내용으로 추론
1. system prompt 생성
2. 문서 검색 합친 내용을 토대로 call_llm 추론

In [ ]:
policy_system_prompt = """
너는 회사 정책자료를 설명하는 챗봇이다.

규칙:
1. 제공된 정책자료를 최우선 근거로 사용한다.
2. 정책자료에 있는 내용을 쉽게 풀어서 설명한다.
3. 정책자료에 없는 내용은 억지로 지어내지 않는다.
4. 답변은 한국어로 한다.
5. 필요하면 "자료상 확인된 내용"과 "일반적인 설명"을 자연스럽게 구분한다.
"""

In [ ]:
def answer_from_policy(question : str, nodes : List[NodeWithScore] ) -> str:
    # 0.5점 이상의 nodes의 문자열을 하나로 합치는 기능
    context = format_nodes( nodes )
    user_content = f"""
        다음은 정책자료 검색 결과다.
        이 자료를 우선 참고해서 답변해라.

        [정책자료]
        {context}

        [현재 사용자 질문]
        {user_question}

        답변 규칙:
        - 정책자료를 바탕으로 쉽게 설명
        - 자료상 확인 가능한 부분을 중심으로 답변
        - 자료에 없는 내용은 억지로 만들지 말 것
        - 필요한 경우 "자료에 따르면" 같은 표현으로 시작해도 좋다
    """
    answer = call_llm(policy_system_prompt, user_content)
    return f"[정책자료 기반 답변]\n{answer}"

In [ ]:
# 추론 결과
answer_from_policy("보안 교육 알려줘?",nodes )

### 아래 내용은 코드만 참고( 이전 내용과 동일한 내용 )

In [ ]:
#히스토리 기능 적용
def call_llm(system_prompt: str, user_content: str) -> str:
    #시스템 msg
    messages = [ChatMessage(role=MessageRole.SYSTEM, content=system_prompt)]
    #이전 대화 기록. assistant
    messages.extend(recent_history_to_messages())
    #사용자 질문 현재 user
    messages.append(ChatMessage(role=MessageRole.USER, content=user_content))

    response = Settings.llm.chat(messages)
    return response.message.content.strip() #strip 양 쪽 공백 삭제
#call_llm(general_answer_prompt, "오늘의 날씨")

In [ ]:
MAX_HISTORY_TURNS = 3
def recent_history_to_messages( max_turns: int = MAX_HISTORY_TURNS ) -> List[ChatMessage]:
    sliced = history[-(max_turns * 2) : ] #짝수 설정 * 2
    messages : List[ChatMessage] = []
    for item in sliced:
        if item['role'] == 'user':
            messages.append(ChatMessage(role=MessageRole.USER, content=item['content']))
        elif item['role'] == 'assistant':
            messages.append(ChatMessage(role=MessageRole.ASSISTANT, content=item['content']))
    return messages
#recent_history_to_messages() #이전 대화 목록 저장 확인

### 최종 오케스트레이터( 이전 내용에 추가 )

In [ ]:
def run_chat(user_question : str ) -> dict:
    # 질문 분류
    route = clean_route(router_chain.invoke({"question": user_question}))
    # policy면 문서 검색
    if route == "policy":
        # 문서에 내용이 있는지 확인
        nodes, top_score = retrieve_policy_docs(user_question)
        if nodes: # 문서에 있는 내용이면 실행
            answer = answer_from_policy(user_question , nodes )
        #fallback(대체수단)
        else: # 문서에 없는 내용인경
            answer = answer_general(
            #기본 일반 상식은 ""처리, fallback로 일반상식인경우 "정책자료...처리"
            user_question, fallback_reason=(
                "정책자료를 먼저 검색했지만, 직접 일치하는 근거가 충분하지 않아 "
                "일반 지식을 바탕으로 답변합니다."
            ),
        )
    else:
        # 추론 후 내용 전달로 수정
        answer = answer_from_general(user_question) #"일반 답변"
    #history 관리
    add_history("user", user_question)
    add_history("assistant", answer)
    save_history(history)

    return {
        "question" : user_question, #사용자 질문
        "route": route, # policy, general 구분된 값
        "answer" : answer # route에 맞는 답변
    }

### test

In [ ]:
examples = [
    "연차 사용 기준이 뭐야?",
    "병가 처리 절차 알려줘",
    "딥러닝과 머신러닝 차이가 뭐야?",
]

for q in examples:
    result = run_chat( q )

    print("\n" + "=" * 70)
    print("질문:", result["question"])
    print("\n답변:")
    print(result["answer"])

In [ ]:
history